<img src="https://sfudial.ca/wp-content/uploads/SFU-DIAL-Logo.png" width=40%>&nbsp;&nbsp;&nbsp;&nbsp;<img src="https://www.sfu.ca/content/dam/sfu/images/brand_extension/SFU-Big-Data_Logo.png" width=40%>

# Lab 5.2: Random Forest Model Building — Vancouver Business Licences

Master AI-enhanced model building techniques using **real City of Vancouver data**. This notebook focuses on Random Forest model training, evaluation, feature importance, and JSON result export.

**Use the TODOs and prompt your AI like a teammate. Think critically, experiment often, and document your process.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

## Lab Outline

- **Part 1:** Environment setup & data loading
- **Part 2:** Data cleaning & exploratory analysis
- **Part 3:** Feature engineering & train/test split
- **Part 4:** Random Forest model, evaluation, feature importance, and result export


## About the Dataset

We are using the **City of Vancouver Business Licences** dataset from the [Vancouver Open Data Portal](https://opendata.vancouver.ca/explore/dataset/business-licences/).

**Problem Statement:** Can we predict whether a Vancouver business licence is currently **active ("Issued")** or **inactive (Cancelled / Gone Out of Business)** based on business characteristics such as type, location, size, and fees?

This is a **binary classification** problem with real-world relevance — understanding what factors contribute to business survival helps city planners, economic development agencies, and entrepreneurs make better decisions.

**Dataset Features:**
- `BusinessType` — Category of business (Restaurant, Office, Retail, etc.)
- `LocalArea` — Vancouver neighbourhood (Downtown, Kitsilano, etc.)
- `NumberOfEmployees` — Reported employee count
- `FeePaid` — Licence fee amount
- `FOLDERYEAR` — Year the licence record belongs to
- `Status` — **Target variable** (Issued vs Inactive)

## Part 1: Environment Setup

In [ ]:
# Install required packages (uncomment if needed)
# !pip install --quiet pandas numpy scikit-learn matplotlib seaborn

In [ ]:
# Core data science libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, RocCurveDisplay)

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")
print("📚 Ready for AI-assisted model building")

## Part 2: Data Loading & Cleaning

### Step 1: Load the Dataset

In [ ]:
# ============================================================
# Load Vancouver Business Licences
# ============================================================
# Data source: https://opendata.vancouver.ca/explore/dataset/business-licences/
#
# Loading order:
#   1. Look for CSV in common project locations (current, data/, parent)
#   2. Download from Vancouver Open Data API as a last resort

import os

CSV_FILENAME = "business-licences.csv"
DATA_URL = (
    "https://opendata.vancouver.ca/api/explore/v2.1/catalog/datasets/"
    "business-licences/exports/csv?limit=20000&delimiter=%2C"
    "&select=folderyear,status,businesstype,localarea,numberofemployees,feepaid"
)

search_paths = [
    CSV_FILENAME,
    os.path.join("data", CSV_FILENAME),
    os.path.join("..", CSV_FILENAME),
    os.path.join("..", "data", CSV_FILENAME),
]

df_raw = None

# --- Try local files first ---
for path in search_paths:
    if os.path.exists(path):
        print(f"📂 Found local file: {path}")
        # Try comma first, then semicolon (Vancouver portal exports use semicolons)
        for sep in [",", ";"]:
            try:
                df_tmp = pd.read_csv(path, sep=sep)
                if df_tmp.shape[1] > 1:
                    df_raw = df_tmp
                    print(f"✅ Loaded {df_raw.shape[0]:,} records with {df_raw.shape[1]} columns from {path}")
                    break
            except Exception:
                continue
        if df_raw is not None:
            break
        else:
            print(f"⚠️  Found {path} but could not parse it — trying next location...")

# --- Download from API if no local file worked ---
if df_raw is None:
    print("📂 No usable local CSV found. Downloading from Vancouver Open Data Portal...")
    try:
        for sep in [";", ","]:
            try:
                df_tmp = pd.read_csv(DATA_URL, sep=sep)
                if df_tmp.shape[1] > 1:
                    df_raw = df_tmp
                    break
            except Exception:
                continue
        if df_raw is not None:
            print(f"✅ Downloaded {df_raw.shape[0]:,} records with {df_raw.shape[1]} columns")
        else:
            raise ValueError("Could not parse downloaded data with any separator")
    except Exception as e:
        print(f"❌ Download failed: {e}")
        print("   Please download the CSV manually from:")
        print("   https://opendata.vancouver.ca/explore/dataset/business-licences/")
        print("   and place it as 'business-licences.csv' in your project or data/ folder.")
        raise FileNotFoundError("Could not load the dataset from any source.")

print(f"\n📊 Columns: {list(df_raw.columns)}")
df_raw.head()

### Step 2: Data Cleaning

In [ ]:
# ============================================================
# Data Cleaning & Preprocessing
# ============================================================

df = df_raw.copy()

# Standardise column names to lowercase
df.columns = df.columns.str.strip().str.lower()

print("📋 Columns available:", list(df.columns))
print(f"\n📊 Shape before cleaning: {df.shape}")

# --- 1. Handle the target variable: Status ---
# Keep only clear outcomes: Issued (active) vs Cancelled / Gone Out of Business (inactive)
valid_statuses = ["Issued", "Cancelled", "Gone Out of Business"]
df = df[df["status"].isin(valid_statuses)].copy()

# Binary target: 1 = Issued (active), 0 = Inactive
df["target"] = (df["status"] == "Issued").astype(int)
print(f"\n🎯 Target distribution:\n{df['target'].value_counts().rename({1:'Active (Issued)', 0:'Inactive'})}")

# --- 2. Clean numeric columns ---
df["numberofemployees"] = pd.to_numeric(df["numberofemployees"], errors="coerce")
df["feepaid"] = pd.to_numeric(df["feepaid"], errors="coerce")
df["folderyear"] = pd.to_numeric(df["folderyear"], errors="coerce")

# Fill missing numerics with median
for col in ["numberofemployees", "feepaid", "folderyear"]:
    df[col] = df[col].fillna(df[col].median())

# Cap extreme employee counts (likely data entry errors)
df["numberofemployees"] = df["numberofemployees"].clip(upper=500)

# --- 3. Clean categorical columns ---
df["businesstype"] = df["businesstype"].fillna("Unknown")
df["localarea"] = df["localarea"].fillna("Unknown")

# Keep only business types with enough samples for modelling
type_counts = df["businesstype"].value_counts()
common_types = type_counts[type_counts >= 20].index
df = df[df["businesstype"].isin(common_types)].copy()

print(f"\n📊 Shape after cleaning: {df.shape}")
print(f"🔍 Missing values remaining: {df[['businesstype','localarea','numberofemployees','feepaid','folderyear']].isnull().sum().sum()}")
print(f"📈 Unique business types: {df['businesstype'].nunique()}")
print(f"📈 Unique local areas: {df['localarea'].nunique()}")

### Step 3: Exploratory Data Analysis

In [ ]:
# ============================================================
# Exploratory Data Analysis (EDA)
# ============================================================
import os
os.makedirs("reports", exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Target distribution
df["target"].value_counts().plot.bar(ax=axes[0, 0], color=["#2ecc71", "#e74c3c"])
axes[0, 0].set_title("Target Distribution (1=Active, 0=Inactive)")
axes[0, 0].set_xticklabels(["Active", "Inactive"], rotation=0)

# 2. Top 10 business types
df["businesstype"].value_counts().head(10).plot.barh(ax=axes[0, 1], color="#3498db")
axes[0, 1].set_title("Top 10 Business Types")
axes[0, 1].invert_yaxis()

# 3. Licence fee distribution by status
for label, group in df.groupby("target"):
    group["feepaid"].clip(upper=2000).plot.hist(
        ax=axes[1, 0], bins=40, alpha=0.5,
        label="Active" if label == 1 else "Inactive"
    )
axes[1, 0].set_title("Fee Paid Distribution by Status")
axes[1, 0].legend()
axes[1, 0].set_xlabel("Fee Paid ($)")

# 4. Employees distribution by status
for label, group in df.groupby("target"):
    group["numberofemployees"].clip(upper=50).plot.hist(
        ax=axes[1, 1], bins=30, alpha=0.5,
        label="Active" if label == 1 else "Inactive"
    )
axes[1, 1].set_title("Number of Employees by Status")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("reports/eda_overview.png", dpi=100, bbox_inches="tight")
plt.show()
print("📊 EDA plots saved to reports/eda_overview.png")

## Part 3: Feature Engineering & Train/Test Split

In [ ]:
# ============================================================
# Feature Engineering
# ============================================================

# Encode categorical variables using LabelEncoder
le_btype = LabelEncoder()
le_area = LabelEncoder()

df["businesstype_enc"] = le_btype.fit_transform(df["businesstype"])
df["localarea_enc"] = le_area.fit_transform(df["localarea"])

# Select features for modelling
feature_cols = ["businesstype_enc", "localarea_enc",
                "numberofemployees", "feepaid", "folderyear"]

X = df[feature_cols].copy()
y = df["target"].copy()

print(f"✅ Feature matrix shape: {X.shape}")
print(f"✅ Target shape: {y.shape}")
print(f"\n📋 Features used: {feature_cols}")
print(f"\n📊 Target balance: {y.value_counts(normalize=True).round(3).to_dict()}")

In [ ]:
# ============================================================
# Train / Test Split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numerical features
scaler = StandardScaler()
num_cols = ["numberofemployees", "feepaid", "folderyear"]
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(f"✅ Training set: {X_train.shape[0]:,} samples")
print(f"✅ Test set:     {X_test.shape[0]:,} samples")
print(f"\n📊 Training target distribution:\n{y_train.value_counts()}")

## Part 4: Model — Random Forest

**Why Random Forest?** It's an ensemble of decision trees that handles non-linear relationships, interactions between features, and provides built-in feature importance — telling us *what drives predictions*.

**AI Prompts to try:**
- *"What are the key hyperparameters for Random Forest?"*
- *"How do I interpret feature importance from a tree-based model?"*

In [ ]:
# ============================================================
# Model 2: Random Forest with Hyperparameter Tuning
# ============================================================
from sklearn.ensemble import RandomForestClassifier

print("🌲 Training Random Forest with GridSearchCV...")

# Define parameter grid
param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)
grid_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring="roc_auc", n_jobs=-1, verbose=0)
grid_rf.fit(X_train, y_train)

print(f"\n✅ Best parameters: {grid_rf.best_params_}")
print(f"✅ Best CV ROC-AUC:  {grid_rf.best_score_:.4f}")

# Predict on test set
y_pred_rf = grid_rf.predict(X_test)
y_prob_rf = grid_rf.predict_proba(X_test)[:, 1]

acc_rf = accuracy_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)

print(f"\n📈 Test Accuracy: {acc_rf:.4f}")
print(f"📈 Test ROC-AUC:  {auc_rf:.4f}")
print(f"\n📋 Classification Report:\n")
print(classification_report(y_test, y_pred_rf, target_names=["Inactive", "Active"]))

In [ ]:
# ============================================================
# Random Forest — Evaluation Visualisations
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1. Confusion Matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Greens", ax=axes[0],
            xticklabels=["Inactive", "Active"],
            yticklabels=["Inactive", "Active"])
axes[0].set_title("Random Forest — Confusion Matrix")
axes[0].set_ylabel("Actual")
axes[0].set_xlabel("Predicted")

# 2. ROC Curve
RocCurveDisplay.from_estimator(grid_rf, X_test, y_test, ax=axes[1], name="Random Forest")
axes[1].set_title("Random Forest — ROC Curve")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random (AUC=0.5)")
axes[1].legend()

# 3. Feature Importance
importances = grid_rf.best_estimator_.feature_importances_
feat_imp = pd.Series(importances, index=feature_cols).sort_values()
feat_imp.plot.barh(ax=axes[2], color="#27ae60")
axes[2].set_title("Random Forest — Feature Importance")

plt.tight_layout()
plt.savefig("reports/random_forest_results.png", dpi=100, bbox_inches="tight")
plt.show()
print("📊 Random Forest results saved to reports/")

In [ ]:
# Save Random Forest results to JSON
import json, os
os.makedirs("reports", exist_ok=True)

rf_results = {
    "model_name": "Random Forest",
    "best_params": {k: (int(v) if isinstance(v, (int, np.integer)) else v)
                    for k, v in grid_rf.best_params_.items()},
    "cv_roc_auc": round(float(grid_rf.best_score_), 4),
    "test_accuracy": round(float(acc_rf), 4),
    "test_roc_auc": round(float(auc_rf), 4),
    "feature_importance": dict(zip(feature_cols,
                                    [round(float(x), 4) for x in grid_rf.best_estimator_.feature_importances_]))
}

with open("reports/model_random_forest_results.json", "w") as f:
    json.dump(rf_results, f, indent=2)

print("💾 Saved: reports/model_random_forest_results.json")
print(json.dumps(rf_results, indent=2))